# RAG and Agentic Memory on Flyte

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/rag-agent-memory/rag-agent-memory-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

A vector store does exactly one thing: it holds a pile of vectors and finds the ones nearest a query vector. No reasoning, nothing that understands your question, nothing that can decline to answer.

Almost everything called "RAG" or "agent memory" is that one operation with different plumbing around it. The difference between them is not the lookup — it is **who holds the pen**:

|  | RAG | Agentic memory |
|---|---|---|
| Who writes | a pipeline, ahead of time | the agent, as it goes |
| What's in it | documents you chose | facts the agent noticed |
| How it's read | embed query → k nearest | embed query → k nearest |

That last row is identical, which is why both live in this one notebook. Steps 0–3 build a read-only index. Step 4 changes who writes to it, and nothing else changes.

**Steps 0, 1 and 3 never call a model**, so you can get to a working retrieval demo — with a picture — before you find your API key.

---

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/rag-agent-memory
    !uv pip install --system -r requirements.txt
    !uv pip install --system keyrings.alt pygments
    !mkdir -p ~/.config/python_keyring && echo -e '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring' > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

from utils.file_viewer import view_file
from utils.report_viewer import show_latest

### Set your Anthropic key

Steps 2, 4 and 5 call a model. **Steps 0, 1 and 3 do not** — run those with no key at all.

Locally you can put `ANTHROPIC_API_KEY=sk-ant-...` in a `.env` file instead; `config.py` loads it for you.

In [ ]:
# Skip this if the key is already in a .env file or your environment.
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

# Want a cheaper model for a workshop? Or a local one?
# os.environ["LLM_MODEL"] = "claude-haiku-4-5"
# os.environ["LLM_PROVIDER"] = "openai"
# os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"   # Ollama

In [ ]:
# Optional: connect to a Flyte or Union cluster.
# Skip this to run everything locally — every step below works with no cluster.
#
# Don't have one? Request demo access at https://union.ai/
#
# !flyte create config \
#     --endpoint <your-endpoint> \
#     --project flytesnacks \
#     --domain development \
#     --builder remote
#
# Then create the secret the tasks read, in the SAME project and domain:
# !flyte create secret ANTHROPIC_API_KEY -p flytesnacks -d development

### Running the steps

Every step is a plain `flyte run`, so the usual flags work:

| | |
|---|---|
| `flyte run --local ...` | Runs on this machine. No cluster, no image build. Still gets caching and reports. |
| `flyte run --local --tui ...` | Same, with the live task UI. |
| `flyte run ...` | Runs on the cluster: containers, retries, and the run graph in the UI. |

The cells below use `--local` so the notebook works with no cluster at all.

Each step calls the ones before it as subtasks. Those are cached, so **only the first run pays for building the index** — everything after starts instantly.

### 0. Build the index

Fetch documents, split them into chunks, embed the chunks, write a Chroma collection. The collection comes out as a `flyte.io.Dir` — one artifact that every later step takes as an input.

The default corpus is **this repository's own tutorial write-ups**: 48 READMEs, about 400 chunks, a few seconds to embed on CPU. Good for a workshop because you can check every answer by opening the file it cites.

No API key needed for this one.

In [ ]:
!flyte run --local step0_index.py index

In [ ]:
# Other corpora, same command:
#   --source flyte-docs                       the Flyte OSS docs (~8MB)
#   --source hf --dataset_repo <hf-dataset>   any HuggingFace text dataset
#   --source local --local_path ~/notes       your own markdown

view_file("step0_index.py", title="fetch, chunk, embed", show_path=True)

### 1. Search it, with no model anywhere

Before letting an LLM near this, look at what retrieval actually *is*: embed the question with the same encoder that embedded the documents, ask the store for the nearest vectors. That's the entire operation.

In [ ]:
!flyte run --local step1_retrieve.py search \
    --question "How do I fine-tune a model with GRPO?"

In [ ]:
show_latest()

Now ask it something the corpus has never heard of.

Watch the similarity scores, not the text.

In [ ]:
!flyte run --local step1_retrieve.py search \
    --question "What is the capital of France?"

In [ ]:
show_latest()

**You still got four chunks.**

That is the most important thing to understand about retrieval, and it's why this step comes before the model. Retrieval has no concept of "I don't know" — it returns the nearest neighbours whether or not they're any good. The similarity scores are the only signal you get. In-corpus questions land around 0.78; that last one was 0.47.

If nothing downstream thresholds on that number, nothing downstream can tell the difference.

In [ ]:
# `retrieve()` is 15 lines and there is no magic in them.
view_file("store.py", title="chunking, embedding, and the lookup", show_path=True)

### 2. Answer from the chunks

Paste the retrieved chunks into the prompt, ask the model to cite them as `[#N]` and to refuse when they don't cover the question.

That's the whole "generation" half. RAG is not an architecture — it's a prompt with freshly-retrieved text in it. Everything hard about it happened upstream, in what you chunked and whether the neighbours were any good.

In [ ]:
!flyte run --local step2_rag_answer.py answer \
    --question "What does the code-mode tutorial teach?"

In [ ]:
# Read the citations here, not in the terminal above — Flyte's console renders
# with Rich, which treats [#1] as markup and swallows it.
show_latest()

Now the same question with retrieval switched off.

The interesting failure is **not** a refusal.

In [ ]:
!flyte run --local step2_rag_answer.py answer \
    --question "What does the code-mode tutorial teach?" --use_retrieval false

In [ ]:
show_latest()

Fluent, confident, and about a tutorial the model has never seen.

Retrieval is what makes the difference *checkable*: every `[#N]` in the grounded answer points at a file you can open and verify.

In [ ]:
view_file("step2_rag_answer.py", title="retrieve, stuff the prompt, answer", show_path=True)

# One file holds every model call, so switching to OpenAI or a local Ollama
# server is an environment variable rather than an edit.
view_file("llm.py", title="the only place a model is called", show_path=True)

### 3. Look at the space

Every chunk is a 384-dimensional vector. UMAP squashes that down to two so it fits on a screen, keeping neighbours near neighbours. The corpus becomes a map with clusters nobody labelled.

Your question goes through the *same fitted* projection and lands as a gold star.

No API key needed here either. The first run takes ~20 seconds while numba compiles; after that the fit is cached and questions are instant.

In [ ]:
!flyte run --local step3_visualize.py visualize \
    --question "How do I fine-tune a model with GRPO?"

In [ ]:
show_latest()

A completely different corner of the corpus:

In [ ]:
!flyte run --local step3_visualize.py visualize \
    --question "brain tumor segmentation"

In [ ]:
show_latest()

And the question the corpus can't answer.

**Read the neighbourhood, not the distance.** UMAP has to put an out-of-corpus question *somewhere*, and it will drop it next to whatever is least unlike it — so a lonely-looking star is not the tell. The tell is that the highlighted chunks have nothing to do with each other or with what you asked, and the scores are low. The numbers are ground truth; the map shows which neighbourhood they came from.

In [ ]:
!flyte run --local step3_visualize.py visualize \
    --question "What is the capital of France?"

In [ ]:
show_latest()

In [ ]:
# The projection is fitted once and cached — deliberately. Refit per question and
# the whole cloud reshuffles between runs, which makes it unreadable. You want the
# map to hold still while the star moves.
view_file("step3_visualize.py", title="fit once, project the query, draw", show_path=True)

### 4. Turn the store around

Same Chroma, same encoder, same nearest-neighbour lookup. The only difference: **the agent writes to it.**

Each turn does four things:

1. embed the message, retrieve the most relevant memories
2. answer, with those memories in the system prompt
3. a second, cheap model call extracts durable facts from the exchange as JSON
4. embed those facts and write them back

The default script is three messages: introduce yourself, mention a constraint, then ask what it knows.

In [ ]:
!flyte run --local step4_memory.py converse

In [ ]:
show_latest()

Turn 3 has **no special handling**. It recalls turn 1 because turn 1 put something in the store that turn 3's question is near. That is the entire mechanism behind "the agent remembers me."

Memory comes back as a `flyte.io.Dir`, so it outlives the run. Feed it to another `converse` and the agent picks up where it left off — copy the path printed above into the cell below.

In [ ]:
# Paste the memory directory from the run above:
# !flyte run --local step4_memory.py converse \
#     --memory_dir <path-from-above> \
#     --messages '["Remind me what my time limit is and who I am."]'

In [ ]:
# Two load-bearing details: extraction is schema-constrained (no regex fishing a
# {...} block out of prose), and near-duplicate memories are dropped so five
# phrasings of the same fact don't crowd out the top-k.
view_file("step4_memory.py", title="retrieve, answer, extract, write back", show_path=True)

### 5. Put it together

Chat on the left, live projection on the right, tabs for retrieved chunks and current memories. Every message retrieves, answers, moves the star, and writes what it learned about you.

In Colab you need `--share` to get a reachable URL. Locally, drop it and open `http://localhost:7860`. The cell blocks while the app runs — stop it to carry on.

In [ ]:
# Uncomment to launch. Startup fits UMAP once, so give it ~30 seconds.
# !python step5_chat_app.py --local --share

On a cluster, this deploys as a real app instead:

```bash
python step5_chat_app.py
```

It mounts the index through `flyte.app.RunOutput`, so the app pod downloads the artifact step 0 already produced rather than rebuilding it.

---

## What you built

- A document index as a cached, versioned Flyte artifact
- Retrieval you can inspect, with the scores that tell you when to distrust it
- Grounded answers whose citations point at files you can open
- A picture of the embedding space, and an honest read of what it does and doesn't show
- An agent that writes to the same store it reads from

**Where to take it next:** re-rank the top 20 down to 4 with a cross-encoder; refuse below a similarity threshold so the "capital of France" case fails honestly; add an `entity_id` to memories for multi-user; add a scheduled task that prunes memories nothing has retrieved in a month.

`embed_and_index` is the only task that knows what Chroma is — point it at pgvector or LanceDB and nothing else changes.